In [11]:
import pandas as pd
import numpy as np
from libcbm.storage import dataframe
from libcbm.model.cbm import stand_cbm_factory
from libcbm.model.cbm import cbm_output
from libcbm.model.cbm import cbm_variables

In [12]:
classifiers = dict(c1=["c1_v1", "c1_v2"], c2=["c2_v1"])


merch_volumes = [
    dict(
        classifier_set=["c1_v1", "?"],
        merch_volumes=[
            dict(
                species="Spruce",
                age_volume_pairs=[[0, 0], [50, 100], [100, 150], [150, 200]],
            )
        ],
    ),
]

In [13]:
n_steps = 100
n_stands = 1
rng = np.random.default_rng()
inventory = dataframe.from_pandas(
    pd.DataFrame(
        dict(
            c1="c1_v1",
            c2="c2_v1",
            admin_boundary="Ontario",
            eco_boundary="Mixedwood Plains",
            age=rng.integers(low=0, high=60, size=n_stands),
            area=1.0,
            delay=0,
            land_class="UNFCCC_FL_R_FL",
            afforestation_pre_type="None",
            historic_disturbance_type="Wildfire",
            last_pass_disturbance_type="Wildfire",
        )
    )
)

CBM Spinup function - initialize the CBM pools and state (cbm_vars)

In [14]:
def spinup(cbm_factory, inventory):
    with cbm_factory.initialize_cbm() as cbm:
        csets, inv = cbm_factory.prepare_inventory(inventory)

        cbm_vars = cbm_variables.initialize_simulation_variables(
            csets,
            inv,
            cbm.pool_codes,
            cbm.flux_indicator_codes,
            inv.backend_type,
        )

        spinup_vars = cbm_variables.initialize_spinup_variables(
            cbm_vars,
            inv.backend_type,
        )
        cbm.spinup(spinup_vars, reporting_func=None)

        cbm_vars = cbm.init(cbm_vars)

        return cbm_vars

In [15]:
def step(cbm_factory, cbm_vars):
    with cbm_factory.initialize_cbm() as cbm:
        cbm_vars = cbm.step_start(cbm_vars)
        cbm_vars = cbm.step_disturbance(cbm_vars)
        cbm_vars = cbm.step_annual_process(cbm_vars)
        cbm_vars = cbm.step_end(cbm_vars)
        return cbm_vars

In [16]:
cbm_factory = stand_cbm_factory.StandCBMFactory(classifiers, merch_volumes)
output = cbm_output.CBMOutput(
    classifier_map=cbm_factory.classifier_value_names,
    disturbance_type_map=cbm_factory.disturbance_types,
)

CBM spinup

In [17]:
cbm_vars = spinup(cbm_factory, inventory)
output.append_simulation_result(0, cbm_vars)

CBM time-stepping

In [18]:
for timestep in range(1, 100):
    if timestep == 90:
        disturbance_types = np.array([4])
    else:
        disturbance_types = np.array([0])

    cbm_vars.parameters["disturbance_type"].assign(disturbance_types)
    step(cbm_factory, cbm_vars)
    # record the end of timestep result
    output.append_simulation_result(timestep, cbm_vars)

extract the results data frames

In [9]:
pools_output = output.pools.to_pandas()
flux_output = output.flux.to_pandas()
state_output = output.state.to_pandas()
parameters_output = output.parameters.to_pandas()
area = output.area.to_pandas()

Plot some output

In [10]:
total_bio = (
    pools_output[
        [
            "timestep",
            "SoftwoodMerch",
            "SoftwoodFoliage",
            "SoftwoodOther",
            "SoftwoodCoarseRoots",
            "SoftwoodFineRoots",
            "HardwoodMerch",
            "HardwoodFoliage",
            "HardwoodOther",
            "HardwoodCoarseRoots",
            "HardwoodFineRoots",
        ]
    ]
    .set_index("timestep")
    .sum(axis=1)
)
total_dom = (
    pools_output[
        [
            "timestep",
            "AboveGroundVeryFastSoil",
            "BelowGroundVeryFastSoil",
            "AboveGroundFastSoil",
            "BelowGroundFastSoil",
            "MediumSoil",
            "AboveGroundSlowSoil",
            "BelowGroundSlowSoil",
            "SoftwoodStemSnag",
            "SoftwoodBranchSnag",
            "HardwoodStemSnag",
            "HardwoodBranchSnag",
        ]
    ]
    .set_index("timestep")
    .sum(axis=1)
)
eco = total_bio + total_dom

total_emissions = (
    flux_output[
        [
            "timestep",
            "DisturbanceBioCO2Emission",
            "DisturbanceBioCH4Emission",
            "DisturbanceBioCOEmission",
            "DisturbanceDOMCO2Emission",
            "DisturbanceDOMCH4Emission",
            "DisturbanceDOMCOEmission",
            "DecayDOMCO2Emission",
        ]
    ]
    .set_index("timestep")
    .sum(axis=1)
)
GrossGrowth_AG = (
    flux_output[
        [
            "timestep",
            "DeltaBiomass_AG",
            "TurnoverMerchLitterInput",
            "TurnoverFolLitterInput",
            "TurnoverOthLitterInput",
        ]
    ]
    .set_index("timestep")
    .sum(axis=1)
)
GrossGrowth_BG = (
    flux_output[
        [
            "timestep",
            "DeltaBiomass_BG",
            "TurnoverCoarseLitterInput",
            "TurnoverFineLitterInput",
        ]
    ]
    .set_index("timestep")
    .sum(axis=1)
)
Gross_growth = GrossGrowth_AG + GrossGrowth_BG
stock_change = eco.diff()
Gross_growth_less_emssions = Gross_growth - total_emissions
products = (
    flux_output[
        [
            "timestep",
            "DisturbanceSoftProduction",
            "DisturbanceHardProduction",
            "DisturbanceDOMProduction",
        ]
    ]
    .set_index("timestep")
    .sum(axis=1)
)
stock_change_and_products = stock_change + products

In [30]:
pd.DataFrame(
    {
        "Biomass": total_bio,
        "DOM": total_dom,
        "Ecosystem": eco,
        "All_Emissions": total_emissions,
        "Gross_growth": Gross_growth,
        "Stock_Change": stock_change,
        "Growth-Emissions": Gross_growth_less_emssions,
        "Stock Change + Products": stock_change_and_products,
    }
).tail(20)

,Biomass,DOM,Ecosystem,All_Emissions,Gross_growth,Stock_Change,Growth-Emissions,Stock Change + Products
timestep,,,,,,,,
80,65.460374,118.298613,183.758987,2.839589,3.151454,0.311865,0.311865,0.311865
81,65.797517,118.277632,184.075149,2.843053,3.159214,0.316162,0.316162,0.316162
82,66.134414,118.261107,184.395521,2.846588,3.166960,0.320373,0.320373,0.320373
83,66.471068,118.248952,184.720020,2.850193,3.174692,0.324499,0.324499,0.324499
84,66.807481,118.241081,185.048562,2.853868,3.182410,0.328542,0.328542,0.328542
85,67.143655,118.237410,185.381065,2.857611,3.190115,0.332503,0.332503,0.332503
86,67.479594,118.237857,185.717450,2.861421,3.197806,0.336385,0.336385,0.336385
87,67.815298,118.242340,186.057638,2.865296,3.205484,0.340188,0.340188,0.340188
88,68.150771,118.250782,186.401552,2.869236,3.213150,0.343914,0.343914,0.343914


In [13]:
flux_output

,identifier,timestep,DisturbanceCO2Production,DisturbanceCH4Production,DisturbanceCOProduction,DisturbanceBioCO2Emission,DisturbanceBioCH4Emission,DisturbanceBioCOEmission,DecayDOMCO2Emission,DisturbanceSoftProduction,...,DisturbanceVFastBGToAir,DisturbanceFastAGToAir,DisturbanceFastBGToAir,DisturbanceMediumToAir,DisturbanceSlowAGToAir,DisturbanceSlowBGToAir,DisturbanceSWStemSnagToAir,DisturbanceSWBranchSnagToAir,DisturbanceHWStemSnagToAir,DisturbanceHWBranchSnagToAir
0,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,1,0.0,0.0,0.0,0.0,0.0,0.0,2.666269,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1,2,0.0,0.0,0.0,0.0,0.0,0.0,2.659014,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1,3,0.0,0.0,0.0,0.0,0.0,0.0,2.652943,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1,4,0.0,0.0,0.0,0.0,0.0,0.0,2.647975,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,1,95,0.0,0.0,0.0,0.0,0.0,0.0,3.760775,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
96,1,96,0.0,0.0,0.0,0.0,0.0,0.0,3.591073,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
97,1,97,0.0,0.0,0.0,0.0,0.0,0.0,3.459494,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
98,1,98,0.0,0.0,0.0,0.0,0.0,0.0,3.355746,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [15]:
total_bio

timestep
0     37.391536
1     38.129237
2     38.864454
3     39.597256
4     40.327706
5     41.055866
6     41.781797
7     42.505557
8     43.227202
9     43.946787
10    44.664364
11    45.379986
12    46.093700
13    46.805554
14    47.515595
15    48.223866
16    48.930411
17    49.635271
18    50.338487
19    51.040097
20    51.740139
21    52.089583
22    52.438648
23    52.787340
24    53.135662
25    53.483618
26    53.831213
27    54.178450
28    54.525334
29    54.871869
30    55.218058
31    55.563906
32    55.909416
33    56.254592
34    56.599438
35    56.943957
36    57.288153
37    57.632030
38    57.975590
39    58.318838
40    58.661776
41    59.004408
42    59.346738
43    59.688768
44    60.030501
45    60.371942
46    60.713092
47     2.853805
48     5.355270
49     7.665176
dtype: float64